### Ventures AI

Chat Bot To query data from [Y-Combinator Startup directory](https://www.ycombinator.com/companies)
> Data Source: https://github.com/yc-oss/api (open sourec Y Combinator companies API)

In [1]:
!uv add -r requirements.txt

Resolved 286 packages in 11ms
Checked 256 packages in 43ms


### Evaluations

Read the ingested companies from Postgres (`ventures_db.yc_oss`, loaded by the `yc_oss_to_ventures_db` Kestra flow)
> Get 1/10th of all records for generating ground truths

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import pandas as pd
import psycopg
from tqdm.auto import tqdm

load_dotenv('../.env')
openai_client = OpenAI()

In [3]:
conn = psycopg.connect(
    host="localhost",
    port=5440,
    dbname="ventures_db",
    user="postgres",
    password="postgres",
)

In [4]:
# sql = """
# insert into yc_oss_sample_records 
# (company_id, title, "content", embedding) 
# select 
# 	yt.company_id,
# 	yt.title,
# 	yt."content",
# 	ye.embedding
# from yc_oss_fulltext yt
# join yc_oss_embeddings ye on  yt.company_id = ye.company_id
# where random() < 0.1
# ;
# """

# conn.execute(sql)

##### Save/Load Sample Questions

In [4]:
# with conn:
df = pd.read_sql("""
    SELECT 
        company_id id,
        title company_name_desc,
        content
    FROM yc_oss_sample_records
    """
, conn)

# conn.close()

print(f"Loaded {len(df)} companies")
print(df.shape)
df.head()

Loaded 521 companies
(521, 3)


/var/folders/m4/y4ly49q978l8dsj7_7n8ts_40000gp/T/ipykernel_34787/3732977826.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


,id,company_name_desc,content
0,12096,Maple Materials — We turn CO2 into battery-gra...,"Maple Materials, Inc. (Richmond, California) d..."
1,451,Credictive — Attribution metatags for online c...,"Industries: B2B, Engineering, Product and Desi..."
2,694,Permutive — Rebuilding data in advertising to ...,As third-party identifiers like the cookie dis...
3,1909,Demeanor.co — Now part of thentwrk.com,"Industries: Consumer, Social | Subindustry: Co..."
4,27491,Sameday — The leading AI workforce for the tra...,Sameday is the AI workforce for the trades — o...


In [5]:
company_records = df.to_dict(orient='records')

In [6]:
company_records[:10]

[{'id': 12096,
  'company_name_desc': 'Maple Materials — We turn CO2 into battery-grade graphite and oxygen.',
  'content': 'Maple Materials, Inc. (Richmond, California) develops a low-cost electrolysis process to split carbon dioxide into graphite and oxygen. Due to several unique features of our production process and product, we have potential to produce battery-grade graphite anode material one-third the cost of current graphite anode materials. When powered by renewables, this process has the potential to be carbon negative.\nIndustries: Industrials, Climate | Subindustry: Industrials -> Climate | Tags: Advanced Materials\nLocation: San Francisco, CA, USA | Regions: United States of America, America / Canada, Remote, Partly Remote\nStage: Early | YC batch: Winter 2019 | Status: Active | Launched: 2018-11-09 | Team size: 6'},
 {'id': 451,
  'company_name_desc': 'Credictive — Attribution metatags for online content.',
  'content': 'Industries: B2B, Engineering, Product and Design | 

#### Generating Ground Truth

In [7]:
prompt_template = """
You emulate an MBA student researching on businesses.
Formulate 5 questions this student might ask based on a company record.
If the content contains the company name, or any identifier to the company, remove the company name/identifier. The questions should NOT contain company identifier (names etc). 
The questions should like a leading question looking at the company description, with a generalized tone
The questions should be complete and neither too short nor too long. If possible, use as fewer words as possible from the record. 

content: {content}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

# output the questions only in the parsable JSON format ["question1", "question2", ..., "question5"]; don't use code blocks


In [8]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response


In [11]:
def generate_questions_ollama(doc):
    prompt = prompt_template.format(**doc)
    prompt += 'ensure to output the questions only in the parsable JSON format ["question1", "question2", ..., "question5"], and that the array format is closed'
    # prompt += 'remove code blocks from output'

    response = ollama.chat(
        model='llama3.2', #'gemma3', #'mistral', #'llama3.2',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.message.content
    return json_response

In [12]:
from concurrent.futures import ThreadPoolExecutor

results_openai = {}
results_llama = {}

with ThreadPoolExecutor(max_workers=2) as executor:
    for doc in tqdm(company_records):
        doc_id = doc['id']
        if doc_id in results_openai and doc_id in results_llama:
            continue

        future_openai = executor.submit(generate_questions, doc)
        future_llama = executor.submit(generate_questions_ollama, doc)

        results_openai[doc_id] = future_openai.result()
        results_llama[doc_id] = future_llama.result()

  0%|          | 0/521 [00:00<?, ?it/s]

In [ ]:
# import json
# # print(f'openai: {results_openai[1754]}\nollama: {results_llama[1754]}')

# # json.loads(results_openai[1754])

# print(results_llama[1754])
# json.loads(results_llama[1754])

In [23]:
# type(results_openai)
# type(results_llama)

#### Save ground truths

In [14]:
import json

def to_rows(results, test=False):
    rows = []
    for (k, v) in results.items():
        try:
            ques = json.loads(v)
            for q in ques:
                rows.append({
                    "company_id": k, "question": q
                })
        except json.JSONDecodeError:
            if test:
                print(f"Skipping company_id {k}: could not parse JSON: {v!r}")
                
            continue

     
    return None if test else rows

##### Test parsing

In [15]:
to_rows(results_openai, True)

In [16]:
to_rows(results_llama, True)

Skipping company_id 23948: could not parse JSON: '[\n  "How does the platform manage large-scale subscription revenue across multiple apps?",\n  "What is the primary benefit of using Superwall\'s paywall product, especially for businesses with high monthly subscriptions?",\n  "How does Superwall handle user retention and churn, particularly through its win-back paywalls and automated refund protection?",\n  "Can you explain how the platform\'s experimentation engine, such as A/B testing, contributes to revenue growth for top-performing apps?",\n  "What are the key differences in pricing between using Superwall\'s subscription infrastructure versus charging a percentage of all subscription revenue?"'
Skipping company_id 29573: could not parse JSON: '{"question1": "How does the new search approach address the limitations of traditional keyword search and other modern AI-based retrieval methods?"}\n{"question2": "What sets this search engine apart from others in terms of its ability to ha

##### Saving CSVs

In [ ]:
# openai_rows = to_rows(results_openai)
# ollama_rows = to_rows(results_llama)

# if len(openai_rows) > 0:
#     ground_truth_df = pd.DataFrame(openai_rows)
#     ground_truth_df.to_csv('data/ground_truths_by_openai.csv')

# if len(ollama_rows) > 0:
#     ground_truth_ollama_df = pd.DataFrame(ollama_rows)
#     ground_truth_ollama_df.to_csv('data/ground_truths_by_ollama.csv')


#### Load ground truths

In [9]:
ground_truth_openaidf = pd.read_csv('data/ground_truths_by_openai.csv')
ground_truth_ollmadf = pd.read_csv('data/ground_truths_by_ollama.csv')
# ground_truth_df.head()

#### Hit-rates and MRR

In [10]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

import importlib
import rag_helper

importlib.reload(rag_helper)
from rag_helper import RAGBase

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

sentence_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

[nltk_data] Downloading package stopwords to /Users/Daudi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##### PgSearch

In [42]:
class RAGPgVector(RAGBase):

    def __init__(self, embedder, conn, **kwargs):
        super().__init__(index=None, **kwargs)
        self.embedder = embedder
        self.conn = conn

    def vec_to_str(self, vector):
        return '[' + ','.join(str(x) for x in vector) + ']'

    def text_search(self, query, num_results=5, sampling=False):
        words = word_tokenize(query)
        stop_words = set(stopwords.words('english'))
        filtered_keywords = [w for w in words if w.isalnum() and w.lower() not in stop_words]
        table = 'yc_oss_sample_records' if sampling else 'yc_oss_fulltext'

        sql = """
            SELECT
                    company_id,
                    title,
                    content,
                    ts_rank(search_vector, query) AS rank
            FROM
                    {table},
                    websearch_to_tsquery('english', '{wapi}') AS query
            WHERE search_vector @@ query
            ORDER BY rank desc
            limit {limits}
        """.format(
            table = table,
            wapi=' or '.join(filtered_keywords), 
            limits=num_results
        )

        rows = self.conn.execute(sql).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'match_words_frequency_rank': r[3]}
            for r in rows
        ]
        

    def vector_search(self, query, num_results=5, sampling=False):
        query_vector = self.embedder.encode(query)
        query_str = self.vec_to_str(query_vector)
        table = 'yc_oss_sample_records' if sampling else 'yc_oss_embeddings'

        rows = self.conn.execute(
            f"""
            SELECT 
                company_id,
                title,
                content,
                1 - (embedding <=> %s::vector) AS cosine_similarity
            FROM {table}
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_str, query_str, num_results)
        ).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'cosine_similarity': r[3]}
            for r in rows
        ]

    def _rrf(self, result_lists: list[list[dict]], k=60, num_results=5):
        scores = {}
        docs = {}

        for results in result_lists:
            for rank, doc in enumerate(results):
                key = doc["company_id"]
                scores[key] = scores.get(key, 0) + 1 / (k + rank)
                docs[key] = doc

        ranked = sorted(scores, key=scores.get, reverse=True)
        return [docs[key] for key in ranked[:num_results]]

    def hybrid_search(self, query: str, num_recs = 5):
        text_results = self.text_search(query, num_results=num_recs)
        vector_results = self.vector_search(query, num_results=num_recs)

        # if len(text_results) == 0 

        return self._rrf([text_results, vector_results], num_results=num_recs)

##### Relevance functions

In [12]:

def compute_single_relevance(q: dict, search_func, sampling=False):
    doc_id = q["company_id"]
    results = search_func(query=q["question"], sampling=sampling)

    relevance = []
    for d in results:
        if d["company_id"] == doc_id:
            # relevance.append(int(d["company_id"] == doc_id))
            if 1 in relevance: 
                relevance.append(0)
            else:
                relevance.append(1)
        else:
            relevance.append(0)


    return relevance

def compute_all_relevance(ground_truth: dict, search_func, sampling=False):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_single_relevance(q, search_func, sampling)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance: list[list[int]]):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance: list[list[int]]):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


##### Search

In [13]:
pconn = psycopg.connect(
        host="localhost",
        port=5440,
        dbname="ventures_db",
        user="postgres",
        password="postgres",
    )

pgIndex = RAGPgVector(
    embedder=sentence_transformer_model,
    conn=pconn,
    llm_client=openai_client,
    # prompt_template=prompt_template,
)

pgIndex_local = RAGPgVector(
    embedder=sentence_transformer_model,
    conn=pconn,
    llm_client_local=ollama,
    # prompt_template=prompt_template,
)

In [25]:
pgIndex.text_search(query='companies building proprietary material', num_results=5, sampling=False)


[{'company_id': 24291,
  'title': 'DigiBuild — We buy & track building materials for construction companies.',
  'content': "Our platform fixes construction's broken supply chain. 70% of construction projects finish late or over budget; procuring and managing building materials is the number 1 reason why. \r\n\r\nWe use LLMs to buy and manage building materials for large construction companies.\r\nWith DigiBuild, construction contractors can find, order, track and manage their materials from the supplier to the job site. We help customers save money, staff time, and improve construction project schedules.\r\n\r\nWe work with the top real estate developers, general contractors, and subcontractors in the US. We manage billions in building material volume annually in the 2nd largest market in the world.\nIndustries: Real Estate and Construction, Construction | Subindustry: Real Estate and Construction -> Construction | Tags: Artificial Intelligence, SaaS, Supply Chain, AI, ML\nLocation: M

In [26]:
pgIndex.vector_search(query='companies building proprietary material', num_results=5, sampling=True)

# pgIndex.vector_search("""
#     How does the company's innovative technology significantly 
#     alter the traditional construction processes to address sustainability challenges?
# """, 3)


[{'company_id': 1753,
  'title': 'SRTX — Powering a better future for textiles.',
  'content': "SRTX builds new materials and software to enable better textiles. SRTX is best known for its first technology, Sheertex, a knit made from one of the world's strongest polymers which has disrupted hosiery through impossibly strong pantyhose. \nIndustries: Industrials, Manufacturing and Robotics | Subindustry: Industrials -> Manufacturing and Robotics | Tags: Hard Tech, Smart Clothing, Consumer, Manufacturing\nLocation: Toronto, ON, Canada; Montreal, QC, Canada | Regions: Canada, America / Canada\nStage: Growth | YC batch: Winter 2018 | Status: Active | Launched: 2017-11-02 | Team size: 150",
  'cosine_similarity': 0.4085072467931803},
 {'company_id': 30592,
  'title': 'Prox — AI technical support for complex physical products',
  'content': "Every product ever sold needs support at some point. That support falls into one of two buckets.\r\n\r\nBucket 1: Simple stuff. T-shirts, screen protecto

###### compute_single_relevance

In [33]:
qObj = {
        'company_id': 1754,
        # 'question': "How does the company's innovative technology significantly alter the traditional construction processes to address sustainability challenges?"
        'question': 'companies building proprietary material'
    }

res = compute_single_relevance(qObj, pgIndex.text_search)
# res = compute_single_relevance(qObj, lambda query=qObj, n=10: pgIndex.text_search(query,n))
res

[0, 0, 1, 0, 0]

In [34]:
qObj = {
        'company_id': 1754,
        'question': "How does the company's innovative technology significantly alter the traditional construction processes to address sustainability challenges?"
        # 'question': 'companies building proprietary material'
    }

res = compute_single_relevance(qObj, pgIndex.vector_search)
# res = compute_single_relevance(qObj, lambda query=qObj, nr=10: pgIndex.vector_search(query,nr))
res

[1, 0, 0, 0, 0]

#### Search Evaluations

##### openai

In [35]:
openai_text_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.text_search)
openai_vector_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.vector_search)

openai_hit_rate_txt = hit_rate(openai_text_relevances)
openai_hit_rate_vec = hit_rate(openai_vector_relevances)

openai_mrr_txt = mrr(openai_text_relevances)
openai_mrr_vec = mrr(openai_vector_relevances)

print(f""""
    Hit rates: Text search: {openai_hit_rate_txt}, Vector search: {openai_hit_rate_vec}\n
    MRR: Text search: {openai_mrr_txt}, Vector search: {openai_mrr_vec}
""")


  0%|          | 0/2605 [00:00<?, ?it/s]

  0%|          | 0/2605 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.28291746641074855, Vector search: 0.3017274472168906

    MRR: Text search: 0.21040307101727426, Vector search: 0.23048624440179147



###### when sampling

In [36]:
sampling = True

openai_text_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.text_search, sampling)
openai_vector_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.vector_search, sampling)

openai_hit_rate_txt = hit_rate(openai_text_relevances)
openai_hit_rate_vec = hit_rate(openai_vector_relevances)

openai_mrr_txt = mrr(openai_text_relevances)
openai_mrr_vec = mrr(openai_vector_relevances)

print(f""""
    Hit rates: Text search: {openai_hit_rate_txt}, Vector search: {openai_hit_rate_vec}\n
    MRR: Text search: {openai_mrr_txt}, Vector search: {openai_mrr_vec}
""")


  0%|          | 0/2605 [00:00<?, ?it/s]

  0%|          | 0/2605 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.4798464491362764, Vector search: 0.5044145873320537

    MRR: Text search: 0.375399872040948, Vector search: 0.4022840690978897



##### ollama

In [37]:
ollama_text_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.text_search)
ollama_vector_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.vector_search)

ollama_hit_rate_txt = hit_rate(ollama_text_relevances)
ollama_hit_rate_vec = hit_rate(ollama_vector_relevances)

ollama_mrr_txt = mrr(ollama_text_relevances)
ollama_mrr_vec = mrr(ollama_vector_relevances)

print(f""""
    Hit rates: Text search: {ollama_hit_rate_txt}, Vector search: {ollama_hit_rate_vec}\n
    MRR: Text search: {ollama_mrr_txt}, Vector search: {ollama_mrr_vec}
""")


  0%|          | 0/2475 [00:00<?, ?it/s]

  0%|          | 0/2475 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.2872727272727273, Vector search: 0.25333333333333335

    MRR: Text search: 0.22344781144781126, Vector search: 0.1926734006734004



###### when sampling

In [38]:
sampling = True

ollama_text_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.text_search, sampling)
ollama_vector_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.vector_search, sampling)

ollama_hit_rate_txt = hit_rate(ollama_text_relevances)
ollama_hit_rate_vec = hit_rate(ollama_vector_relevances)

ollama_mrr_txt = mrr(ollama_text_relevances)
ollama_mrr_vec = mrr(ollama_vector_relevances)

print(f""""
    Hit rates: Text search: {ollama_hit_rate_txt}, Vector search: {ollama_hit_rate_vec}\n
    MRR: Text search: {ollama_mrr_txt}, Vector search: {ollama_mrr_vec}
""")

  0%|          | 0/2475 [00:00<?, ?it/s]

  0%|          | 0/2475 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.41494949494949496, Vector search: 0.4072727272727273

    MRR: Text search: 0.3442087542087545, Vector search: 0.3259461279461283



> #### Hit Rate/MRR Interpretation
> This results are as expected because of the reasons below:
>   - The ground truth questions are built from a subset of the database (1/10th) - hit-rate and mrr performance improves on checking the sampled dataset vs checking against the full dataset
>       - Testing on the main data includes other similar companies that answer the questions in the ground truth
>   - The results we're looking for are more generalized (top k matching) than, for example, specific company documents with specific information, where the results are required to be specific (top 1 matching)
> <br/>
> &nbsp;

##### RRF

In [13]:
def rrf(result_lists: list[list[dict]], k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = doc["company_id"]
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query: str, num_recs = 5):
    text_results = pgIndex.text_search(query, num_results=num_recs)
    vector_results = pgIndex.vector_search(query, num_results=num_recs)

    # if len(text_results) == 0 

    return rrf([text_results, vector_results], num_results=num_recs)



In [14]:
query = 'companies building proprietary material'

res = hybrid_search(query)

res


[{'company_id': 25810,
  'title': "Material Depot — India's Fastest Growing Home Decor destination for Floor & Wall Decor",
  'content': 'Material Depot is reimagining home interiors for the next generation of Indian homeowners. We bring design-forward materials: tiles, laminates, wall panels & more - straight from manufacturers to customers, cutting out the old, broken supply chain. Our online engagement fuels real-time trend insights, enabling us to launch new designs faster, in smaller batches, and at unbeatable prices. The result: beautiful homes built smarter, quicker, and more affordably. With over 10,000 homes served and a ',
  'cosine_similarity': 0.46236670017242765},
 {'company_id': 24291,
  'title': 'DigiBuild — We buy & track building materials for construction companies.',
  'content': "Our platform fixes construction's broken supply chain. 70% of construction projects finish late or over budget; procuring and managing building materials is the number 1 reason why. \r\n\r\

#### LLM Monitoring

In [15]:
pgIndex.rag(query, res)

'Good sir, of the companies before us, these seem the nearest fit to thy quest for **proprietary material**:\n\n1. **Mighty Buildings** — Most apt of all.  \n   They have wrought **a new material of their own devising**, fit for 3D-printing whole dwellings; not mere stone nor concrete, but a substance bespoke for their craft. In this, proprietary matter doth clearly dwell.\n\n2. **Material Depot** — A merchant of design-forward materials.  \n   Though it doth not proclaim a secret matter of its own making, it curates tiles, laminates, wall panels, and more, and bringeth them swift from maker unto buyer. Its strength lies more in **distribution and design** than in proprietary substance.\n\n3. **Axal** — A smith of custom printed boards.  \n   It designeth, sources, and quality-tests custom PCBs, yet the materials themselves are not shown as uniquely its own. Its gift is in **service and orchestration**, not in the invention of a new material.\n\n4. **DigiBuild** — A master of procureme

In [ ]:
pgIndex_local.rag(query, res)

##### Logfire Monitoring

In [44]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext
import logfire

# from pydantic_ai.models.openai import OpenAIChatModel
# from pydantic_ai.providers.openai import OpenAIProvider

# ollama_model = OpenAIChatModel(
#     model_name='llama3.2',
#     provider=OpenAIProvider(base_url='http://localhost:11434/v1', api_key='ollama'),
# )

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider

# Initialize the Ollama model pointing to your local endpoint
ollama_model = OllamaModel(
    model_name='llama3.2',
    provider=OllamaProvider(base_url='http://localhost:11434/v1')
)


@dataclass
class SearchDeps:
    index: RAGPgVector

yc_startup_agent = Agent(
    # 'openai:gpt-5.4-mini',
    ollama_model,
    deps_type=SearchDeps,
    instructions=pgIndex.instructions,
)

@yc_startup_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    # ctx.deps.index is the minsearch index we injected via SearchDeps
    res = ctx.deps.index.hybrid_search(query)

    print(res)

    return res

# logfire.configure()
# logfire.instrument_pydantic_ai()

deps = SearchDeps(index=pgIndex)

swali = 'companies building proprietary material'
result = await yc_startup_agent.run(swali, deps=deps)

result.output

Traceback (most recent call last):
  File "/Users/Daudi/Documents/Daudi/Projects/Data Engineering/LLM Zoomcamp/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3746, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/m4/y4ly49q978l8dsj7_7n8ts_40000gp/T/ipykernel_34787/1546067790.py", line 49, in <module>
    result = await yc_startup_agent.run(swali, deps=deps)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Daudi/Documents/Daudi/Projects/Data Engineering/LLM Zoomcamp/.venv/lib/python3.12/site-packages/pydantic_ai/agent/abstract.py", line 551, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Daudi/Documents/Daudi/Projects/Data Engineering/LLM Zoomcamp/.venv/lib/python3.12/site-packages/pydantic_ai/run.py", line 402, in next
    return await self._run_node_with_hooks(node, self._advance_graph)
           ^^^^

#### LLM Evaluation
##### Faithfulness - RAGAs

The major usecase for LLM is summarization of the results from the hybrid search. We'll evaluate for major hallucinations, and whether the answer captures/summarizes accurately the retrieval results

In [ ]:
INSTRUCTIONS = '''
from the context, which are results from a retrieval, check for accuracy of the provided answer. The answer is in poetic/shakespearian english.
Classify the answers as RELEVANT, PARTLY-RELEVANT, NON-RELEVANT, 
'''

PROMPT_TEMPLATE = '''
QUESTION:
{swali}

CONTEXT:
{context}

answer:
{result.output}
'''.strip()